<!-- SPDX-FileCopyrightText: 2026 Orbital Research Cluster for Celestial Applications (ORCCA) Lab, University of Colorado at Boulder -->
<!-- SPDX-License-Identifier: ISC -->

# Finite-Burn Propagation Demo
---
*Scarabaeus OD Framework | Last revised 2026*

This notebook demonstrates how to propagate a mission sequence that contains finite-burn arcs in **Scarabaeus**.

The goal is to show the complete workflow:

1. define one or more `Maneuver` objects,
2. attach a maneuver to `ForceModelTranslation` using `finite_burn=True`,
3. build a `MissionSequence` with coast and finite-burn arcs,
4. propagate the sequence,
5. inspect mass evolution and trajectory continuity.

The examples below are intentionally propagation-focused. Orbit-determination and covariance analysis can be added in a separate notebook once the basic finite-burn workflow is clear.


## 0. Imports and Setup

We first import Scarabaeus, the scientific Python stack, load the tutorial data, load SPICE kernels, and define the units and frames used throughout the notebook.


In [1]:
import os
import sys
import numpy as np
import matplotlib.pyplot as plt

import scarabaeus as scb
from scarabaeus import (
    EpochArray,
    Spacecraft,
    ArrayWUnits,
    StateArray,
    SpiceManager,
    MissionSequence,
    CelestialBody,
    Propagator,
    ManeuverParser,
    ArrayWFrame,
    Maneuver,
    StateDefinition,
    ForceModelTranslation,
)

# ── working directory ─────────────────────────────────────────────
# Notebooks are expected to be run from the repository root or from tutorials/.
if os.path.basename(os.getcwd()) == "tutorials":
    repo_root = os.path.dirname(os.getcwd())
else:
    repo_root = os.getcwd()

if os.path.join(repo_root, "tutorials") not in sys.path:
    sys.path.insert(0, os.path.join(repo_root, "tutorials"))

import supplementary as supp

data = supp.load_data()

# Prefer the tutorial metakernel when available.
scb.SpiceManager.clear_kernels()
try:
    scb.SpiceManager.load_kernel_from_mkfile(data.mk.path)
except Exception:
    SpiceManager.load_kernel("data/kernels/locked/lsk/naif0012.tls")

# ── units and frames ──────────────────────────────────────────────
kg, sec, N, km, rad = scb.Units.get_units(["kg", "sec", "N", "km", "rad"])
J2000, ITRF93, ECLIPJ2000, IAUEARTH = scb.Frame.generate_common_frames()

plt.rcParams.update({
    "figure.figsize": (9, 5),
    "axes.grid": True,
    "font.size": 11,
})


SCB tutorial data up to date.


## 1. Helper Functions

The helper functions below keep the finite-burn sections compact. The important implementation detail is that each mission arc gets its own `EpochArray`. This avoids using one global time array for coast and burn arcs with different start/end times.


In [2]:
def make_time_array(t0, tf, dt_sec):
    """Create a TDB EpochArray from t0 to tf, including the final epoch."""
    return EpochArray(np.arange(t0, tf + dt_sec, dt_sec), sys="TDB")


def print_arc(name, t_start, t_end):
    """Print an arc window in UTC and hours."""
    try:
        t0_str = SpiceManager.et2utc(t_start, "C", 3)
        tf_str = SpiceManager.et2utc(t_end, "C", 3)
    except Exception:
        t0_str = f"{t_start:.3f} ET"
        tf_str = f"{t_end:.3f} ET"

    duration_hr = (t_end - t_start) / 3600.0
    print(f"{name}")
    print(f"  Start   : {t0_str}")
    print(f"  End     : {tf_str}")
    print(f"  Duration: {duration_hr:.3f} hr")


def empty_pv_state(epoch, spacecraft, origin, frame):
    """Create an empty position/velocity state used by sequence arcs after the first arc."""
    return StateArray(
        epoch=EpochArray(epoch, sys="TDB"),
        origin=origin,
        state=StateDefinition.empty_pv(spacecraft, frame=frame),
    )


def plot_mass_history(mission, spacecraft, title="Mass Evolution"):
    """Plot spacecraft mass along the propagated mission sequence."""
    epochs = mission.total_epochsTDB.times.values
    mass_vals = np.array([spacecraft.mass(t).values for t in epochs], dtype=float)

    plt.figure()
    plt.plot(epochs, mass_vals)
    plt.xlabel("TDB seconds past J2000 [s]")
    plt.ylabel("Mass [kg]")
    plt.title(title)
    plt.tight_layout()
    plt.show()

    print(f"Initial mass: {mass_vals[0]:.6f} kg")
    print(f"Final mass  : {mass_vals[-1]:.6f} kg")
    print(f"Mass used   : {mass_vals[0] - mass_vals[-1]:.6f} kg")


## 2. Define Finite-Burn Maneuvers Manually

A `Maneuver` contains the thrust magnitude, mass-flow rate, start/end times, and thrust direction. In this first example the burns are simple constant-parameter burns.

The thrust direction should be normalized. Here we use a direction halfway between +x and +z.


In [3]:
# Direction halfway between +x and +z.
u = 1.0 / np.sqrt(2.0)

# Example electric-propulsion-like mass flow from thrust and Isp.
g0 = 9.80665       # [m/s^2]
Isp = 2200.0       # [s]
thrust_value = 0.200  # [N]
mdot = thrust_value / (Isp * g0)  # [kg/s]

M1 = Maneuver()
M1.thrust = ArrayWUnits(thrust_value, N)
M1.mass_flow = ArrayWUnits(mdot, kg / sec)
M1.start_time = EpochArray(923000647.3506516, "TDB")
M1.end_time = EpochArray(923004047.3506516, "TDB")
M1.ux = ArrayWUnits(u, None)
M1.uy = ArrayWUnits(0.0, None)
M1.uz = ArrayWUnits(u, None)

M2 = Maneuver()
M2.thrust = ArrayWUnits(thrust_value, N)
M2.mass_flow = ArrayWUnits(mdot, kg / sec)
M2.start_time = EpochArray(923004047.3506516, "TDB")
M2.end_time = EpochArray(923007447.3506516, "TDB")
M2.ux = ArrayWUnits(u, None)
M2.uy = ArrayWUnits(0.0, None)
M2.uz = ArrayWUnits(u, None)

for name, man in [("M1", M1), ("M2", M2)]:
    print_arc(name, man.start_time.times.values, man.end_time.times.values)


M1
  Start   : 2029 APR 01 09:02:58.165
  End     : 2029 APR 01 09:59:38.165
  Duration: 0.944 hr
M2
  Start   : 2029 APR 01 09:59:38.165
  End     : 2029 APR 01 10:56:18.165
  Duration: 0.944 hr


## 3. Spacecraft, Central Body, and Initial State

The finite burn changes the spacecraft translational dynamics through the thrust acceleration and changes the spacecraft mass through `mass_flow`.


In [4]:
# Spacecraft properties
Orbiter_drymass = ArrayWUnits(2300.0, kg)
Orbiter_area = ArrayWUnits(1e-6, km**2)
Orbiter_cr_srp = ArrayWUnits(1.5, None)

Orbiter = Spacecraft(
    "Orbiter",
    -60000,
    Orbiter_drymass,
    Orbiter_area,
    Orbiter_cr_srp,
)

origin = CelestialBody.from_constants("SUN")
frame = J2000

# Start one hour before the first burn.
margin_before_M1 = 3600.0  # [s]
start_time = M1.start_time.times.values - margin_before_M1

# Example heliocentric initial state.
r0 = np.array([-2.6561672815488e08, 1.8837080839972e08, 1.0199307996915e08])  # [km]
v0 = np.array([-9.0779264528807, -12.4823165310294, -4.4599484708606])        # [km/s]

initial_state = StateArray(
    epoch=EpochArray(start_time, sys="TDB"),
    origin=origin,
    state=(
        StateDefinition()
        .position(Orbiter, ArrayWFrame(ArrayWUnits(r0, km), J2000))
        .velocity(Orbiter, ArrayWFrame(ArrayWUnits(v0, km / sec), J2000))
    ),
)


## 4. Build Coast and Finite-Burn Arcs

A mission sequence is built arc by arc:

- a coast leg from the initial epoch to the first burn start,
- finite-burn arc 1,
- finite-burn arc 2,
- a final coast leg.

For each burn arc, the force model is created with `finite_burn=True` and the corresponding `maneuver` object.


In [5]:
dt = ArrayWUnits(1.0, sec)
dt_sec = float(dt.values)

# ── Coast before M1 ──────────────────────────────────────────────
coast_0_start = start_time
coast_0_end = M1.start_time.times.values
coast_0_duration = coast_0_end - coast_0_start

time_array_coast_0 = make_time_array(coast_0_start, coast_0_end, dt_sec)
force_model_coast_0 = ForceModelTranslation(primary_body=Orbiter, cannonball_SRP=True)
prop_model_coast_0 = Propagator(
    primary_body=Orbiter,
    state_vector=initial_state,
    tspan=time_array_coast_0,
    force_models=force_model_coast_0,
)

# ── Finite burn M1 ───────────────────────────────────────────────
M1_start = M1.start_time.times.values
M1_end = M1.end_time.times.values
M1_duration = M1_end - M1_start

time_array_fb_M1 = make_time_array(M1_start, M1_end, dt_sec)
state_fb_M1 = empty_pv_state(M1_start, Orbiter, origin, J2000)
force_model_fb_M1 = ForceModelTranslation(
    primary_body=Orbiter,
    cannonball_SRP=True,
    finite_burn=True,
    maneuver=M1,
)
prop_model_fb_M1 = Propagator(
    primary_body=Orbiter,
    state_vector=state_fb_M1,
    tspan=time_array_fb_M1,
    force_models=force_model_fb_M1,
)

# ── Finite burn M2 ───────────────────────────────────────────────
M2_start = M2.start_time.times.values
M2_end = M2.end_time.times.values
M2_duration = M2_end - M2_start

time_array_fb_M2 = make_time_array(M2_start, M2_end, dt_sec)
state_fb_M2 = empty_pv_state(M2_start, Orbiter, origin, J2000)
force_model_fb_M2 = ForceModelTranslation(
    primary_body=Orbiter,
    cannonball_SRP=True,
    finite_burn=True,
    maneuver=M2,
)
prop_model_fb_M2 = Propagator(
    primary_body=Orbiter,
    state_vector=state_fb_M2,
    tspan=time_array_fb_M2,
    force_models=force_model_fb_M2,
)

# ── Final coast after M2 ─────────────────────────────────────────
coast_final_start = M2_end
coast_final_end = M2_end + 3400.0
coast_final_duration = coast_final_end - coast_final_start

time_array_coast_final = make_time_array(coast_final_start, coast_final_end, dt_sec)
state_coast_final = empty_pv_state(coast_final_start, Orbiter, origin, J2000)
force_model_coast_final = ForceModelTranslation(primary_body=Orbiter, cannonball_SRP=True)
prop_model_coast_final = Propagator(
    primary_body=Orbiter,
    state_vector=state_coast_final,
    tspan=time_array_coast_final,
    force_models=force_model_coast_final,
)

for name, t0, tf in [
    ("Initial coast", coast_0_start, coast_0_end),
    ("Finite burn M1", M1_start, M1_end),
    ("Finite burn M2", M2_start, M2_end),
    ("Final coast", coast_final_start, coast_final_end),
]:
    print_arc(name, t0, tf)


Initial coast
  Start   : 2029 APR 01 08:02:58.165
  End     : 2029 APR 01 09:02:58.165
  Duration: 1.000 hr
Finite burn M1
  Start   : 2029 APR 01 09:02:58.165
  End     : 2029 APR 01 09:59:38.165
  Duration: 0.944 hr
Finite burn M2
  Start   : 2029 APR 01 09:59:38.165
  End     : 2029 APR 01 10:56:18.165
  Duration: 0.944 hr
Final coast
  Start   : 2029 APR 01 10:56:18.165
  End     : 2029 APR 01 11:52:58.165
  Duration: 0.944 hr


## 5. Assemble and Propagate the Mission Sequence

`MissionSequence.addLeg` is used for ordinary coast arcs. `MissionSequence.addFiniteBurn` is used for burn arcs. The sequence propagator passes the terminal state of one arc into the next arc, so the later arc state definitions can be initialized as empty position/velocity states.


In [6]:
MS = MissionSequence("FiniteBurnDemo")

MS.addLeg(
    "Initial coast",
    ArrayWUnits(coast_0_duration, sec),
    initial_state,
    prop_model_coast_0,
    dt,
)

MS.addFiniteBurn(
    "Finite burn M1",
    ArrayWUnits(M1_duration, sec),
    state_fb_M1,
    prop_model_fb_M1,
    dt,
    None,
)

MS.addFiniteBurn(
    "Finite burn M2",
    ArrayWUnits(M2_duration, sec),
    state_fb_M2,
    prop_model_fb_M2,
    dt,
    None,
)

MS.addLeg(
    "Final coast",
    ArrayWUnits(coast_final_duration, sec),
    state_coast_final,
    prop_model_coast_final,
    dt,
)

mission = MS.propagate()
print("Propagation complete.")
print(f"Number of propagated epochs: {len(mission.total_epochsTDB.times.values)}")



--- Propagating Segment 1/7: 'Initial coast' [Leg] ---

[IC] segment='Initial coast' type='Leg' epoch=922997047.3506516 sec (TDB)
  position     n= 3 sid=-60000 frame=J2000 vals[:6]=[-2.65616728e+08  1.88370808e+08  1.01993080e+08]
  velocity     n= 3 sid=-60000 frame=J2000 vals[:6]=[ -9.07792645 -12.48231653  -4.45994847]

                            Starting propagation...                             


Integrating:   0%|                                                        | 0.00/3600.00 s [00:00<?]

Integrating:   0%|                                                        | 0.00/3600.00 s [00:00<?]

Integrating:   0%|                                                    | 0.01/3600.00 s [00:00<14:11]

Integrating:   0%|                                                        | 0.00/3600.00 s [00:00<?]

...

Integrating: 100%|█████████████████████████████████████████████████| 3400.00/3400.00 s [00:00<00:00]

Integrating: 100%|█████████████████████████████████████████████████| 3400.00/3400.00 s [00:00<00:00]

Integrating: 100%|█████████████████████████████████████████████████| 3400.00/3400.00 s [00:00<00:00]


 =================== DOP853 integration complete. ==================
Propagation complete.

[GLOBAL STATE after 'Final coast']
  key=('position', 3, -60000, 'J2000', 0) -> global[0:3] = [-2.65741916e+08  1.88198493e+08  1.01931503e+08]
  key=('velocity', 3, -60000, 'J2000', 0) -> global[3:6] = [ -9.0652612  -12.49099597  -4.46423055]


Propagation complete.
Number of propagated epochs: 13801


## 7. Plot Trajectory Components

This is a simple sanity check on the propagated trajectory. The objective is not to validate the force model physically, but to verify that the sequence propagated continuously across coast and burn arcs.


In [7]:
# The exact trajectory attribute names may differ depending on the current Scarabaeus branch.
# This helper tries common patterns and raises a useful error if the branch API changed.
def get_cartesian_history(mission):
    candidates = [
        "total_state",
        "total_states",
        "state",
        "states",
        "trajectory",
    ]
    for attr in candidates:
        if hasattr(mission, attr):
            obj = getattr(mission, attr)
            try:
                vals = obj.values
            except Exception:
                vals = obj
            arr = np.asarray(vals)
            if arr.ndim == 2 and arr.shape[1] >= 6:
                return arr[:, :6]
            if arr.ndim == 2 and arr.shape[0] >= 6:
                return arr[:6, :].T
    raise AttributeError(
        "Could not infer the Cartesian state history from this mission object. "
        "Inspect dir(mission) and update get_cartesian_history for this branch."
    )

try:
    X = get_cartesian_history(mission)
    t = mission.total_epochsTDB.times.values
    hours = (t - t[0]) / 3600.0

    plt.figure()
    plt.plot(hours, X[:, 0], label="x")
    plt.plot(hours, X[:, 1], label="y")
    plt.plot(hours, X[:, 2], label="z")
    plt.xlabel("Time since start [hr]")
    plt.ylabel("Position [km]")
    plt.title("Propagated Position Components")
    plt.legend()
    plt.tight_layout()
    plt.show()

    plt.figure()
    plt.plot(hours, X[:, 3], label="vx")
    plt.plot(hours, X[:, 4], label="vy")
    plt.plot(hours, X[:, 5], label="vz")
    plt.xlabel("Time since start [hr]")
    plt.ylabel("Velocity [km/s]")
    plt.title("Propagated Velocity Components")
    plt.legend()
    plt.tight_layout()
    plt.show()
except Exception as err:
    print(err)


Could not infer the Cartesian state history from this mission object. Inspect dir(mission) and update get_cartesian_history for this branch.


## 8. Parse Finite Burns from a Maneuver Specification File

Instead of manually creating `Maneuver` objects, a maneuver specification file can be parsed using `ManeuverParser`. This is the workflow used when finite burns come from a reference trajectory or an external mission-design tool.


In [8]:
spec_path = os.path.join(
    repo_root,
    "data",
    "dynamic_setup",
    "thruster_coefficients",
    "example.mission_maneuver_spec",
)

if os.path.exists(spec_path):
    with open(spec_path, "r") as handle:
        parsed_maneuvers = ManeuverParser()._parse_maneuver_spec(handle)

    print(f"Parsed {len(parsed_maneuvers)} finite burns from:")
    print(spec_path)

    for i, man in enumerate(parsed_maneuvers[:5]):
        print_arc(f"Parsed burn {i}", man.start_time.times.values, man.end_time.times.values)
else:
    parsed_maneuvers = []
    print(f"Maneuver specification file not found: {spec_path}")


Maneuver specification file not found: /Users/zael5647/scarabaeus/docs/online_documentation/sphinx_files/_collections/data/dynamic_setup/thruster_coefficients/example.mission_maneuver_spec


## 9. Optional: Fit a Polynomial Maneuver Profile

Finite-burn parameters may also be represented by polynomial coefficients. A common workflow is:

1. parse a list of short burns,
2. use the midpoint of each burn as the independent variable,
3. fit polynomial coefficients for thrust, mass flow, and thrust direction components,
4. create a new `Maneuver` whose fields contain those polynomial coefficients.

This section only builds and plots the fitted profile. Use it when you want a smoothed finite-burn representation instead of a list of piecewise maneuvers.


In [9]:
if parsed_maneuvers:
    thrust_list = np.array([float(m.thrust.convert_to(N).values) for m in parsed_maneuvers])
    mass_flow_list = np.array([float(m.mass_flow.values) for m in parsed_maneuvers])
    ux_list = np.array([float(m.ux.values) for m in parsed_maneuvers])
    uy_list = np.array([float(m.uy.values) for m in parsed_maneuvers])
    uz_list = np.array([float(m.uz.values) for m in parsed_maneuvers])

    start_times = np.array([float(m.start_time.times.values) for m in parsed_maneuvers])
    end_times = np.array([float(m.end_time.times.values) for m in parsed_maneuvers])
    mid_times = 0.5 * (start_times + end_times)

    t0_poly = float(start_times[0])
    tf_poly = float(end_times[-1])
    t_eval = np.linspace(t0_poly, tf_poly, 500)

    # Use a safe order if fewer than 7 points are available.
    order = min(6, len(parsed_maneuvers) - 1)

    fit_data = {
        "Thrust [N]": thrust_list,
        "Mass flow [kg/s]": mass_flow_list,
        "ux [-]": ux_list,
        "uy [-]": uy_list,
        "uz [-]": uz_list,
    }

    coeffs = {}
    for label, y in fit_data.items():
        coeffs[label] = np.polyfit(mid_times, y, order)
        poly = np.poly1d(coeffs[label])

        plt.figure()
        plt.plot(mid_times, y, ".", label="Parsed values")
        plt.plot(t_eval, poly(t_eval), label=f"Order-{order} fit")
        plt.xlabel("TDB seconds past J2000 [s]")
        plt.ylabel(label)
        plt.title(f"Polynomial Fit: {label}")
        plt.legend()
        plt.tight_layout()
        plt.show()
else:
    print("No parsed maneuvers available; skipping polynomial fit.")


No parsed maneuvers available; skipping polynomial fit.


## 10. Notes and Common Pitfalls

- Use one `EpochArray` per arc. A finite-burn arc should use a time grid that spans that burn, not a coast arc grid.
- Use `ForceModelTranslation(..., finite_burn=True, maneuver=M)` for a burn arc.
- Use `MissionSequence.addFiniteBurn(...)` for burn arcs and `MissionSequence.addLeg(...)` for coast arcs.
- If the burn arc is not the first arc, initialize its position/velocity with `StateDefinition.empty_pv(...)` or branch-equivalent empty state definitions. The sequence will fill the initial state from the previous arc.
- Keep thrust units in `N`, mass flow in `kg/sec`, and translational states in `km` and `km/sec` unless the branch explicitly expects otherwise.
- For direction components, prefer normalized unit vectors. If direction errors are estimated later, keep the nominal direction normalized and estimate small angular corrections separately.
